# Context-Adjusted Receiver Separation

This notebook documents the first baseline for the receiver separation project. The research question is:

> Given the situation a route runner is in, how much separation does he create or preserve compared with what an average NFL route runner would be expected to create in the same situation?

The initial target is intentionally narrow:

```text
delta_sep = separation_at_pass_release - separation_at_snap
SOE_route = delta_sep_actual - E[delta_sep | observable snap context]
```

That framing avoids claiming total receiver value. It only measures route-level separation creation or preservation from snap to pass release.

In [1]:
from pathlib import Path
import csv
import statistics
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from separation_over_expected.models import evaluate_models, fit_models, split_rows
from separation_over_expected.reports import dataset_overview, read_csv_rows, receiver_summary_rows

ROUTE_TABLE = PROJECT_ROOT / "data" / "processed" / "route_level_snap_to_release.csv"
PREDICTIONS = PROJECT_ROOT / "data" / "processed" / "route_level_baseline_predictions.csv"
METRICS = PROJECT_ROOT / "data" / "processed" / "baseline_metrics.csv"
RECEIVER_SUMMARY = PROJECT_ROOT / "data" / "processed" / "receiver_baseline_summary.csv"

PROJECT_ROOT

PosixPath('/Users/moggo/Projects/NFL/Separation_Over_Expected')

## Data Shape

The route table has one row per route runner on plays where we observe both a snap event and a pass-forward event. It includes targeted and untargeted route runners because `pffScoutingData.csv` identifies every player with `pff_role == "Pass Route"`.

In [2]:
rows = read_csv_rows(ROUTE_TABLE)
overview = dataset_overview(rows)
overview

{'routes': 35032, 'plays': 7583, 'players': 472, 'weeks': [1, 2, 3, 4, 5, 6, 7, 8], 'split_rows': {'train': 27084, 'validation': 3696, 'test': 4252}}

In [3]:
def count_by(rows, column):
    counts = {}
    for row in rows:
        counts[row[column]] = counts.get(row[column], 0) + 1
    return sorted(counts.items(), key=lambda item: (-item[1], item[0]))

count_by(rows, "officialPosition")[:10]

[('WR', 20415), ('TE', 8015), ('RB', 6224), ('FB', 347), ('QB', 27), ('C', 3), ('T', 1)]

## Target Behavior

`delta_sep` is often negative. That is not automatically bad route running. Many receivers start with defensive cushion at the snap, then defenders close space before the throw. The model's job is to learn how much closing is expected from the starting context.

In [4]:
def describe(values):
    values = sorted(values)
    def q(p):
        return values[int((len(values) - 1) * p)]
    return {
        "min": round(min(values), 3),
        "p05": round(q(0.05), 3),
        "median": round(q(0.50), 3),
        "mean": round(statistics.fmean(values), 3),
        "p95": round(q(0.95), 3),
        "max": round(max(values), 3),
    }

for column in ["sep_snap", "sep_release", "delta_sep", "route_depth", "time_to_throw_frames"]:
    print(column, describe([float(row[column]) for row in rows]))

sep_snap {'min': 0.502, 'p05': 2.055, 'median': 5.936, 'mean': 5.995, 'p95': 10.626, 'max': 23.087}
sep_release {'min': 0.135, 'p05': 0.765, 'median': 2.91, 'mean': 3.742, 'p95': 9.954, 'max': 42.174}
delta_sep {'min': -21.305, 'p05': -6.783, 'median': -2.221, 'mean': -2.253, 'p95': 2.666, 'max': 23.483}
route_depth {'min': -10.32, 'p05': 0.45, 'median': 7.43, 'mean': 8.658, 'p95': 19.4, 'max': 54.43}
time_to_throw_frames {'min': 10.0, 'p05': 18.0, 'median': 27.0, 'mean': 29.472, 'p95': 50.0, 'max': 192.0}


## Baseline Models

We start with transparent baselines before adding heavier modeling libraries:

- `global_mean`: every route gets the training-set mean
- `smoothed_group_mean`: shrinkage mean by official position, alignment, and man/zone label
- `ridge_context`: regularized linear model using numeric snap context and one-hot categorical football context

The first split trains on weeks 1-6, validates on week 7, and tests on week 8.

In [5]:
splits = split_rows(rows)
models = fit_models(splits["train"])
metrics = evaluate_models(models, splits)
metrics

[{'model': 'global_mean', 'split': 'train', 'rows': '27084', 'mae': '2.214', 'rmse': '2.914', 'r2': '0.000', 'bias': '0.000', 'mean_actual': '-2.231', 'mean_predicted': '-2.231', 'n': '27084'}, {'model': 'smoothed_group_mean', 'split': 'train', 'rows': '27084', 'mae': '2.157', 'rmse': '2.835', 'r2': '0.054', 'bias': '-0.003', 'mean_actual': '-2.231', 'mean_predicted': '-2.228', 'n': '27084'}, {'model': 'ridge_context', 'split': 'train', 'rows': '27084', 'mae': '1.710', 'rmse': '2.366', 'r2': '0.341', 'bias': '0.000', 'mean_actual': '-2.231', 'mean_predicted': '-2.231', 'n': '27084'}, {'model': 'global_mean', 'split': 'validation', 'rows': '3696', 'mae': '2.273', 'rmse': '2.983', 'r2': '-0.000', 'bias': '-0.043', 'mean_actual': '-2.273', 'mean_predicted': '-2.231', 'n': '3696'}, {'model': 'smoothed_group_mean', 'split': 'validation', 'rows': '3696', 'mae': '2.219', 'rmse': '2.916', 'r2': '0.044', 'bias': '-0.048', 'mean_actual': '-2.273', 'mean_predicted': '-2.226', 'n': '3696'}, {'mode

The useful comparison is not whether the model is perfect. The useful comparison is whether context explains more than a naive average. If the context model reduces error on week 8, we have a workable first definition of expected separation creation.

In [6]:
for row in metrics:
    if row["split"] in {"validation", "test"}:
        print(f"{row['model']:>20} {row['split']:>10}  R2={row['r2']}  RMSE={row['rmse']}  MAE={row['mae']}")

         global_mean validation  R2=-0.000  RMSE=2.983  MAE=2.273
 smoothed_group_mean validation  R2=0.044  RMSE=2.916  MAE=2.219
       ridge_context validation  R2=0.308  RMSE=2.481  MAE=1.791
         global_mean       test  R2=-0.003  RMSE=2.904  MAE=2.198
 smoothed_group_mean       test  R2=0.045  RMSE=2.834  MAE=2.142
       ridge_context       test  R2=0.327  RMSE=2.380  MAE=1.721


## First Receiver-Level View

For the first leaderboard, `SOE_route` is the residual from the ridge context model. This should be treated as exploratory. It does not yet include player uncertainty, position-specific models, route clusters, or coverage-responsibility modeling.

In [7]:
scoring_model = next(model for model in models if model.name == "ridge_context")
overall = receiver_summary_rows(rows, scoring_model, min_routes=50)
overall[:15]

[{'nflId': '53478', 'displayName': 'Rondale Moore', 'officialPosition': 'WR', 'routes': '97', 'mean_soe': '1.849', 'median_soe': '1.108', 'total_soe': '179.350'}, {'nflId': '44975', 'displayName': 'Jeremy McNichols', 'officialPosition': 'RB', 'routes': '79', 'mean_soe': '1.413', 'median_soe': '1.568', 'total_soe': '111.602'}, {'nflId': '53464', 'displayName': 'Javonte Williams', 'officialPosition': 'RB', 'routes': '91', 'mean_soe': '1.221', 'median_soe': '1.061', 'total_soe': '111.071'}, {'nflId': '52440', 'displayName': 'Clyde Edwards-Helaire', 'officialPosition': 'RB', 'routes': '62', 'mean_soe': '1.107', 'median_soe': '1.425', 'total_soe': '68.616'}, {'nflId': '44912', 'displayName': 'Jonnu Smith', 'officialPosition': 'TE', 'routes': '88', 'mean_soe': '1.101', 'median_soe': '0.261', 'total_soe': '96.890'}, {'nflId': '30842', 'displayName': 'Marcedes Lewis', 'officialPosition': 'TE', 'routes': '50', 'mean_soe': '1.066', 'median_soe': '0.148', 'total_soe': '53.303'}, {'nflId': '52733'

A pooled leaderboard mixes different jobs. Running backs, tight ends, and wide receivers face different route distributions and coverage structures, so the next view separates wide receivers.

In [8]:
wr_summary = receiver_summary_rows(rows, scoring_model, min_routes=50, position="WR")
wr_summary[:20]

[{'nflId': '53478', 'displayName': 'Rondale Moore', 'officialPosition': 'WR', 'routes': '97', 'mean_soe': '1.849', 'median_soe': '1.108', 'total_soe': '179.350'}, {'nflId': '42437', 'displayName': 'Ty Montgomery', 'officialPosition': 'WR', 'routes': '70', 'mean_soe': '0.478', 'median_soe': '0.362', 'total_soe': '33.491'}, {'nflId': '46522', 'displayName': 'Byron Pringle', 'officialPosition': 'WR', 'routes': '120', 'mean_soe': '0.474', 'median_soe': '0.092', 'total_soe': '56.914'}, {'nflId': '47839', 'displayName': 'Mecole Hardman', 'officialPosition': 'WR', 'routes': '201', 'mean_soe': '0.449', 'median_soe': '-0.160', 'total_soe': '90.231'}, {'nflId': '52442', 'displayName': 'Michael Pittman', 'officialPosition': 'WR', 'routes': '218', 'mean_soe': '0.387', 'median_soe': '0.161', 'total_soe': '84.463'}, {'nflId': '52622', 'displayName': 'Freddie Swain', 'officialPosition': 'WR', 'routes': '147', 'mean_soe': '0.354', 'median_soe': '-0.117', 'total_soe': '52.054'}, {'nflId': '46279', 'dis

## Next Questions

The baseline is good enough to move from feasibility into research. The next useful improvements are:

1. Fit and report position-specific models for WR, TE, and RB.
2. Add uncertainty intervals for receiver summaries so small samples do not masquerade as signal.
3. Replace nearest defender with a coverage-aware assignment or weighted defender distance.
4. Cluster route shapes using the normalized snap-to-release movement features.
5. Validate whether early-season SOE predicts later-season separation or receiving outcomes.